In [137]:
import customfunctions as ctf
import pandas as pd
import numpy as np
import heartpy as hp
import neurokit2 as nk
import prepro as prep  # Custom functions for preprocessing
import os
import glob
from pathlib import Path
from scipy.ndimage import uniform_filter1d
from sklearn.preprocessing import MinMaxScaler, RobustScaler

# Parameters

### Preprocessing parameters

In [138]:
# BVP preprocessing
bvp_prepro_winsz = 10
bvp_prepro_ovlap = 0.02
bvp_fs = 64

# EDA preprocessing
eda_fs = 4
iter_num = 7

### Processing parameters

In [139]:
basalscene = 0
targetscene = 4
# Note, subject 10 does not have scene 2 nor 3 data
subjects = [1, 2, 3, 4, 6, 8, 9]
winsz = 90
ovlap = 0.5
# Outlier removal method, either iqr_outlier, mod_zscore_outlier (both applicable in non-normal data) or winsorization (winsor_outlier)
outliermeth = 'winsor_outlier'
# Outlier treatment strategy, either elimination ('elim') or imputation ('impute')
outliertreatment = 'impute'
p_low=0.5
p_high=0.95
# Normalization type, either z-score ('zscore'), mean ('mean'), minmax scaling ('minmax'), robust scaling ('robust')
normtype = 'zscore'
base_dir = Path().resolve()

# Data import function

In [140]:
folderpath = os.path.join(base_dir, "..", "RawData")
folderpath = os.path.abspath(folderpath)


def dataimport(subjects, folderpath, scene):
    bvp_rawdata_list = []
    eda_rawdata_list = []

    for subject in subjects:
        subject_str = f"S{subject}" if subject == 10 else f"S{subject:02d}"
        pattern = os.path.join(
            folderpath,
            f"S{subject}",
            "Empatica",
            f"P300_{subject_str}R0{scene}*",
            "Raw",
        )
        matched_folders = glob.glob(pattern)
        raw_folder = matched_folders[0]
        bvp_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileBVP.csv")).drop(
            "Datetime", axis=1
        )
        eda_dataholder = pd.read_csv(os.path.join(raw_folder, r"fileEDA.csv")).drop(
            "Datetime", axis=1
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_rawdata_list.append(bvp_dataholder)
        eda_rawdata_list.append(eda_dataholder)

    bvp_rawdata_df = pd.concat(bvp_rawdata_list, ignore_index=True)
    eda_rawdata_df = pd.concat(eda_rawdata_list, ignore_index=True)

    return bvp_rawdata_df, eda_rawdata_df


basal_bvp_rawdata, basal_eda_rawdata = dataimport(
    subjects, folderpath, scene=basalscene
)
tscene_bvp_rawdata, tscene_eda_rawdata = dataimport(
    subjects, folderpath, scene=targetscene
)

# Preprocessing function

In [141]:
def preprocess(subjects, bvp_df, eda_df, bvp_fs, eda_fs, winsz, ovlap, iter_num):
    bvp_prepdata_list = []
    eda_prepdata_list = []
    for subject in subjects:
        bvp_dataholder = pd.DataFrame(
            prep.preprocess_bvp(
                sig=bvp_df[bvp_df["Subject"] == subject]["valueBVP"],
                fs=bvp_fs,
                winsz=winsz,
                ovlap=ovlap,
            ),
            columns=["valueBVP"],
        )
        eda_dataholder = pd.DataFrame(
            prep.preprocess_eda(
                sig=eda_df[eda_df["Subject"] == subject]["valueEDA"],
                fs=eda_fs,
                iter_num=iter_num,
                verbose=False,
            )[0],
            columns=["valueEDA"],
        )

        bvp_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(bvp_dataholder)
        )
        eda_dataholder.insert(
            loc=0, column="Subject", value=[subject] * len(eda_dataholder)
        )

        bvp_prepdata_list.append(bvp_dataholder)
        eda_prepdata_list.append(eda_dataholder)

    bvp_prepdata_df = pd.concat(bvp_prepdata_list, ignore_index=True)
    eda_prepdata_df = pd.concat(eda_prepdata_list, ignore_index=True)

    return bvp_prepdata_df, eda_prepdata_df


basal_bvp_prepdata, basal_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=basal_bvp_rawdata,
    eda_df=basal_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)

tscene_bvp_prepdata, tscene_eda_prepdata = preprocess(
    subjects=subjects,
    bvp_df=tscene_bvp_rawdata,
    eda_df=tscene_eda_rawdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    winsz=bvp_prepro_winsz,
    ovlap=bvp_prepro_ovlap,
    iter_num=iter_num,
)


# Processing function

In [142]:
import warnings
def get_bvp_feats(window, smooth_window, fs):
    wd = {}
    wd = hp.peakdetection.detect_peaks(
        window, smooth_window, ma_perc=20, sample_rate=fs
    )
    wd = hp.analysis.calc_rr(wd["peaklist"], sample_rate=fs, working_data=wd)
    wd = hp.peakdetection.check_peaks(
        wd["RR_list"], wd["peaklist"], window[wd["peaklist"]], working_data=wd
    )
    wd = hp.analysis.clean_rr_intervals(working_data=wd, method="quotient-filter")
    rr_list = wd["RR_list_cor"]
    rr_diff = np.diff(rr_list)
    rr_sqdiff = np.power(rr_diff, 2)
    wd, msrs = hp.analysis.calc_ts_measures(
        rr_list, rr_diff, rr_sqdiff, working_data=wd
    )
    wd, msrs = hp.analysis.calc_fd_measures(measures=msrs, working_data=wd)
    bvp_feats = pd.DataFrame([msrs])[
        [
            "bpm",
            "sdnn",
            'rmssd',
            "pnn50",
            "hr_mad",
            "lf",
            "hf",
            "lf/hf",
            "p_total",
            "lf_nu",
            "hf_nu",
        ]
    ]

    return bvp_feats


def get_eda_feats(window, fs):
    signals, info = nk.eda_process(window, fs)
    mean_eda = np.nanmean(window)
    mean_tonic = np.nanmean(signals["EDA_Tonic"])
    scr_peak_count = np.nansum(signals["SCR_Peaks"])
    scr_sum_amp = np.nansum(info["SCR_Amplitude"])
    scr_mean_amp = np.nanmean(info["SCR_Amplitude"])
    scr_mean_risetime = np.nanmean(info["SCR_RiseTime"])
    scr_mean_recoverytime = np.nanmean(info["SCR_RecoveryTime"])

    eda_feats = pd.DataFrame(
        [
            {
                "mean_eda": mean_eda,
                "mean_tonic": mean_tonic,
                "scr_peak_count": scr_peak_count,
                "scr_sum_amp": scr_sum_amp,
                "scr_mean_amp": scr_mean_amp,
                "scr_mean_risetime": scr_mean_risetime,
                "scr_mean_recoverytime": scr_mean_recoverytime,
            }
        ]
    )

    return eda_feats


def process(subjects, bvp_prepdata_df, eda_prepdata_df, bvp_fs, eda_fs, ovlap, winsz):
    feat_mat_list = []
    for subject in subjects:
        # BVP
        bvp_probe = np.array(
            bvp_prepdata_df[bvp_prepdata_df["Subject"] == subject]["valueBVP"]
        )
        bvp_smooth = uniform_filter1d(
            bvp_probe, size=int(0.75 * bvp_fs), mode="nearest"
        )
        bvp_windowed = ctf.timewindowpadded(
            data=bvp_probe, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        smooth_windowed = ctf.timewindowpadded(
            data=bvp_smooth, fs=bvp_fs, ovlap=ovlap, winsz=winsz
        )
        bvp_holder_list = []
        for windnum in range(0,bvp_windowed.shape[0]):
            bvp_feat_row = get_bvp_feats(
                window=bvp_windowed[windnum],
                smooth_window=smooth_windowed[windnum],
                fs=bvp_fs,
            )
            bvp_holder_list.append(bvp_feat_row)
        bvp_holder_df = pd.concat(bvp_holder_list, ignore_index=True)

        # EDA
        eda_probe = np.array(
            eda_prepdata_df[eda_prepdata_df["Subject"] == subject]["valueEDA"]
        )
        eda_windowed = ctf.timewindowpadded(
            data=eda_probe, fs=eda_fs, ovlap=ovlap, winsz=winsz
        )
        eda_holder_list = []
        for windnum in range(0,eda_windowed.shape[0]):
            eda_feat_row = get_eda_feats(window=eda_windowed[windnum], fs=eda_fs)
            eda_holder_list.append(eda_feat_row)
        eda_holder_df = pd.concat(eda_holder_list, ignore_index=True)

        valid_len = min(len(bvp_holder_df), len(eda_holder_df))
        joined_df = pd.concat([bvp_holder_df.iloc[:valid_len], eda_holder_df.iloc[:valid_len]], axis=1)
        joined_df.insert(loc=0, column="Subject", value=[subject] * len(joined_df))
        feat_mat_list.append(joined_df)
        
    feat_mat_df = pd.concat(feat_mat_list, ignore_index=True)
    return feat_mat_df


basal_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=basal_bvp_prepdata,
    eda_prepdata_df=basal_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

tscene_feat_mat = process(
    subjects=subjects,
    bvp_prepdata_df=tscene_bvp_prepdata,
    eda_prepdata_df=tscene_eda_prepdata,
    bvp_fs=bvp_fs,
    eda_fs=eda_fs,
    ovlap=ovlap,
    winsz=winsz,
)

warnings.filterwarnings('ignore')

In [143]:
basal_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,62.466476,60.745158,58.491992,0.136986,39.0625,1.020411e+03,1.069850e+03,0.953789,2.602937e+03,48.817396,51.182604,-0.005557,-0.005470,17,0.133718,0.007866,0.455882,0.750000
1,1,63.840399,74.530595,67.652525,0.126582,46.8750,9.577659e+02,8.816200e+02,1.086370,2.133140e+03,52.069872,47.930128,-0.000625,-0.000620,17,0.109199,0.006825,0.593750,0.950000
2,1,67.908512,101.847943,86.004657,0.121622,62.5000,2.391600e+03,2.279704e+03,1.049084,8.190353e+03,51.197694,48.802306,0.002401,0.002226,18,0.096585,0.005681,1.132353,1.100000
3,1,68.746803,86.251777,62.814230,0.145161,62.5000,8.017548e+02,1.627016e+03,0.492776,2.820222e+03,33.010728,66.989272,0.009608,0.009637,15,0.122990,0.008199,1.050000,1.027778
4,2,71.431793,83.631418,54.698887,0.102041,46.8750,1.013593e+03,8.258658e+02,1.227310,2.730259e+03,55.102794,44.897206,-0.023166,-0.023263,74,0.429396,0.005882,0.414384,0.535714
5,2,74.250000,78.236675,58.271328,0.091837,46.8750,5.973643e+02,8.025838e+02,0.744301,4.640375e+03,42.670461,57.329539,0.001783,0.001801,78,0.461093,0.005911,0.442308,0.632353
6,2,77.111562,85.465727,64.847496,0.102041,62.5000,6.725265e+02,3.094232e+02,2.173484,2.605237e+03,68.488895,31.511105,0.002389,0.001928,26,0.402167,0.015468,0.557692,0.972222
7,2,85.404504,106.789867,111.293637,0.189873,54.6875,2.977515e+03,1.315640e+03,2.263169,6.425113e+03,69.354939,30.645061,-0.000033,0.000183,22,0.376104,0.017096,0.579545,0.966667
8,3,81.923703,56.356810,50.081731,0.128205,31.2500,8.102969e+02,1.374918e+03,0.589342,2.323080e+03,37.080880,62.919120,0.000878,0.000861,35,0.156139,0.004461,0.414286,0.550000
9,3,85.128820,55.584121,51.950588,0.100000,31.2500,5.423301e+02,8.034705e+02,0.674984,1.635130e+03,40.297951,59.702049,-0.000713,-0.000727,36,0.211838,0.005884,0.416667,0.479167


In [144]:
tscene_feat_mat

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,81.836066,152.535356,183.818015,0.400000,132.8125,555.034557,8813.351106,0.062977,9.368386e+03,5.924549,94.075451,-0.045856,-0.046815,8,1.216692,0.173813,0.500000,0.500000
1,1,82.781641,162.364117,198.894241,0.300000,125.0000,2637.181814,21458.035604,0.122899,2.409522e+04,10.944835,89.055165,0.002018,0.001967,9,0.780578,0.086731,0.916667,2.687500
2,1,95.506003,162.295716,209.194185,0.321429,109.3750,328.932239,10734.888424,0.030641,1.106382e+04,2.973044,97.026956,-0.000685,-0.000626,15,0.194791,0.012986,0.916667,1.386364
3,1,83.666416,150.505660,178.519105,0.357143,125.0000,6156.922797,2618.510736,2.351307,8.775434e+03,70.160896,29.839104,-0.007803,-0.007733,13,0.377604,0.029046,0.942308,1.700000
4,2,78.580079,75.979394,76.874659,0.185567,46.8750,12298.684488,102159.925499,0.120387,4.864561e+08,10.745093,89.254907,0.028943,0.029459,3,1.203188,0.401063,0.333333,1.250000
5,2,75.445068,73.168328,68.916007,0.154639,46.8750,1704.778655,707.243015,2.410457,4.536405e+03,70.678414,29.321586,0.024070,0.024029,5,1.778431,0.355686,0.450000,1.250000
6,2,74.540253,62.494373,70.965130,0.159574,31.2500,1474.510816,967.347004,1.524283,2.623368e+03,60.384794,39.615206,0.006219,0.006048,5,0.630056,0.126011,0.450000,6.250000
7,2,79.868637,118.150189,130.902317,0.226667,39.0625,2975.379926,5331.133430,0.558114,8.559546e+03,35.819842,64.180158,0.048503,0.045097,4,0.701856,0.233952,2.166667,1.250000
8,3,83.105462,92.884511,107.248187,0.112903,31.2500,854.456830,2388.034460,0.357808,3.242491e+03,26.351862,73.648138,0.017701,0.018153,9,0.866724,0.096303,1.194444,2.142857
9,3,81.280868,74.733513,105.539186,0.090909,31.2500,932.651988,942.916541,0.989114,2.015015e+03,49.726362,50.273638,-0.156093,-0.157092,30,0.426845,0.014228,0.633333,0.541667


In [145]:
def remove_outliers_iqr(col, treatment='elim'):
    q1 = col.quantile(0.25)
    q3 = col.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    if treatment == 'impute':
        return col.clip(lower=lower, upper=upper)
    else:
        return col[(col >= lower) & (col <= upper)]

def remove_outliers_modz(col, treatment='elim'):
    x_mid = col.median()
    mad = abs(col - x_mid).median()
    mz = 0.6745 * (col - x_mid) / mad

    if treatment == 'impute':
        return col.mask(abs(mz) > 3.5, x_mid)
    else:
        return col[abs(mz) <= 3.5]

def remove_outliers_winsor(col, p_low=0.05, p_high=0.95):
    lower = col.quantile(p_low)
    upper = col.quantile(p_high)
    return col.clip(lower=lower, upper=upper)


def norm_with_ref(subjects, feat_mat, ref_mat, normtype=str, outliermeth=outliermeth, outliertreatment=outliertreatment):
        # This line is redundant, eliminate and substitute application on this cell based on preference.
    channel_cols = [
        col for col in ref_mat.columns if col not in ["Subject"]
    ]

    normdata_list = []

    # Data is normalized per subject   ####### Corregir
    for subject in subjects:
        data = feat_mat[feat_mat["Subject"] == subject].copy()
        
        if outliermeth == 'iqr_outlier':
            def outlier_func(col):
                return remove_outliers_iqr(col, outliertreatment)
        elif outliermeth == 'mod_zscore_outlier':
            def outlier_func(col):
                return remove_outliers_modz(col, outliertreatment)
        elif outliermeth == 'winsor_outlier':
            def outlier_func(col):
                return remove_outliers_winsor(col, p_low=0.5, p_high=0.95)

        data[channel_cols] = data[channel_cols].apply(outlier_func)

        ref_data = ref_mat[ref_mat["Subject"] == subject].drop("Subject", axis=1)
        ref_data = ref_data.apply(outlier_func).dropna()

        if normtype == 'zscore':
            ref_mean = np.nanmean(ref_data)
            ref_std = np.nanstd(ref_data)
            data[channel_cols] = (data[channel_cols] - ref_mean) / ref_std
        elif normtype == 'mean':
            ref_mean = np.nanmean(ref_data)
            data[channel_cols] = (data[channel_cols] - ref_mean) / ref_mean
        elif normtype == 'minmax':
            scaler = MinMaxScaler(feature_range=(-0.5,0.5))
            scaler.fit(ref_data[channel_cols])
            scaled = scaler.transform(data[channel_cols])
            #scaled = np.clip(scaled, 0, 1)
            data[channel_cols] = scaled
        elif normtype == 'robust':
            scaler = RobustScaler()
            scaler.fit(ref_data[channel_cols])
            data[channel_cols] = scaler.transform(data[channel_cols])
        normdata_list.append(data.dropna())
    normdata_df = pd.concat(normdata_list, ignore_index=True)

    return normdata_df

norm = norm_with_ref(subjects=subjects, feat_mat=tscene_feat_mat, ref_mat=basal_feat_mat, normtype=normtype)
norm

,Subject,bpm,sdnn,rmssd,pnn50,hr_mad,lf,hf,lf/hf,p_total,lf_nu,hf_nu,mean_eda,mean_tonic,scr_peak_count,scr_sum_amp,scr_mean_amp,scr_mean_risetime,scr_mean_recoverytime
0,1,-0.294365,-0.225873,-0.194539,-0.370833,-0.249668,1.102309,8.652137,-0.371111,9.060170,-0.363410,-0.284347,-0.371200,-0.371200,-0.361041,-0.370134,-0.371048,-0.370350,-0.369772
1,1,-0.294365,-0.221314,-0.187580,-0.370883,-0.255798,2.063414,17.953650,-0.371083,20.068618,-0.361092,-0.286665,-0.371195,-0.371195,-0.361041,-0.370476,-0.371116,-0.370350,-0.368852
2,1,-0.284666,-0.221367,-0.179498,-0.370883,-0.255798,1.102309,9.539106,-0.371111,9.842772,-0.363410,-0.282031,-0.371197,-0.371197,-0.357626,-0.370662,-0.371143,-0.370350,-0.369772
3,1,-0.293957,-0.225873,-0.194539,-0.370867,-0.255798,4.825384,8.652137,-0.369334,9.060170,-0.314625,-0.286665,-0.371200,-0.371200,-0.359195,-0.370662,-0.371143,-0.370330,-0.369627
4,2,-0.294669,-0.297019,-0.296210,-0.365518,-0.323323,9.485376,78.835122,-0.364745,373690.876439,-0.322213,-0.288421,-0.365660,-0.365659,-0.361619,-0.364599,-0.365330,-0.365279,-0.364556
5,2,-0.296086,-0.298290,-0.298881,-0.365530,-0.323323,1.749164,2.480445,-0.363628,5.552056,-0.303206,-0.318783,-0.365662,-0.365662,-0.361167,-0.364157,-0.365364,-0.365279,-0.364556
6,2,-0.296086,-0.298290,-0.298881,-0.365530,-0.326853,1.749164,2.480445,-0.364308,5.552056,-0.311113,-0.318783,-0.365662,-0.365662,-0.361167,-0.364825,-0.365419,-0.365279,-0.360715
7,2,-0.293679,-0.264624,-0.254707,-0.365487,-0.326853,2.323318,4.452334,-0.364745,7.370016,-0.322213,-0.307683,-0.365645,-0.365647,-0.361619,-0.364825,-0.365419,-0.363960,-0.364556
8,3,-0.349129,-0.324768,-0.304153,-0.416482,-0.388650,0.293668,1.923821,-0.416310,2.643350,-0.395648,-0.356105,-0.416580,-0.416580,-0.409440,-0.415905,-0.416518,-0.415645,-0.414923
9,3,-0.349129,-0.324768,-0.304153,-0.416482,-0.388650,0.324746,1.923821,-0.415883,2.643350,-0.379855,-0.356105,-0.416608,-0.416608,-0.395252,-0.416080,-0.416520,-0.415682,-0.415551


In [146]:
channel_cols = [col for col in basal_feat_mat.columns if col not in ["Subject"]]
basal_outlier_treated = basal_feat_mat
basal_outlier_treated[channel_cols] = basal_feat_mat[channel_cols].apply(remove_outliers_winsor, args=(0.1,0.9,))

In [147]:
tscene_outlier_treated = tscene_feat_mat
tscene_outlier_treated[channel_cols] = tscene_feat_mat[channel_cols].apply(remove_outliers_winsor, args=(0.1,0.9,))

In [148]:
outpath = os.path.join(base_dir, 'Outputs', f'outlier_{outliertreatment}', outliermeth)
basal_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{basalscene}_nonnorm.csv'), index=False)
tscene_outlier_treated.to_csv(os.path.join(outpath,'non-normalized',rf'biometric_feat_mat_scene_{targetscene}_nonnorm.csv'), index=False)
norm.to_csv(os.path.join(outpath, rf'biometric_feat_mat_scene_{targetscene}_{normtype}.csv'), index=False)